In [1]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 1 — Run cell ini SENDIRI, tunggu kernel restart
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)


{'status': 'ok', 'restart': True}

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.environ['DATASET_25'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'
os.environ['DATASET_15'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'

print(os.path.exists(os.environ['DATASET_25']))
print(os.path.exists(os.environ['DATASET_15']))

sys.path.append('/content')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
True


In [3]:
sys.path.append('/content/drive/MyDrive')

In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 2 — Hapus cache + tulis common.py (step_sec=1)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import shutil, os, sys, base64

OUTDIR = 'outputs_lstm_sensitivity'
if os.path.exists(OUTDIR):
    shutil.rmtree(OUTDIR)
    print(f'✅ Cache dihapus: {OUTDIR}/')
else:
    print('ℹ️  Belum ada cache')

_b64 = (
    'IiIiCmNvbW1vbi5weSDigJQgU2hhcmVkIHBpcGVsaW5lIGZ1bmN0aW9ucyBmb3IgdGhlIHJldmlz'
        'aW9uIGV4cGVyaW1lbnRzLgoKVGhlc2UgZnVuY3Rpb25zIGFyZSBjb3BpZWQgVkVSQkFUSU0gZnJv'
        'bSB0aGUgYXV0aG9yJ3Mgbm90ZWJvb2tzCihBTVNURV8yNW1zLmlweW5iIC8gQU1TVEVfMTVtcy5p'
        'cHluYikgc28gdGhhdCBldmVyeSByZXZpc2lvbiBleHBlcmltZW50CnVzZXMgZXhhY3RseSB0aGUg'
        'c2FtZSBwc2V1ZG8tZXBpc29kZSBjb25zdHJ1Y3Rpb24sIGVwaXNvZGUgc3BsaXQsIHdpbmRvd2lu'
        'ZywKdGFidWxhci1mZWF0dXJlIGV4dHJhY3Rpb24sIGFuZCBtZXRyaWMgY29tcHV0YXRpb24gYXMg'
        'dGhlIG9yaWdpbmFsIHBhcGVyLgpEbyBOT1QgbW9kaWZ5IHRoZXNlIHVubGVzcyB5b3UgYWxzbyBy'
        'ZS1ydW4gdGhlIG1haW4gcmVzdWx0cy4KClR3byBkYXRhc2V0cyBhcmUgdXNlZCBieSB0aGUgb3Jp'
        'Z2luYWwgY29kZToKICAgIDI1IG0vcyAgLT4gIERhdGFzZXRfMS5jc3YKICAgIDE1IG0vcyAgLT4g'
        'IERhdGFzZXRfMi5jc3YKU2V0IHRoZSBwYXRocyBpbiBDT05GSUcgYmVsb3cgKG9yIHBhc3MgdGhl'
        'bSB0byBsb2FkX2RhdGFzZXQoKSkuCiIiIgppbXBvcnQgb3MsIGpzb24sIGdsb2IsIHJhbmRvbQpp'
        'bXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHNrbGVhcm4ucHJlcHJv'
        'Y2Vzc2luZyBpbXBvcnQgU3RhbmRhcmRTY2FsZXIKZnJvbSBza2xlYXJuLm1ldHJpY3MgaW1wb3J0'
        'IChhY2N1cmFjeV9zY29yZSwgcHJlY2lzaW9uX3JlY2FsbF9mc2NvcmVfc3VwcG9ydCwKICAgICAg'
        'ICAgICAgICAgICAgICAgICAgICAgICBjb25mdXNpb25fbWF0cml4KQoKIyDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBDT05GSUcgICh2ZXJiYXRpbSB2'
        'YWx1ZXMgZnJvbSB0aGUgbm90ZWJvb2tzKQojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgApTRUVEUyAgICAgICAgICA9IFs0MiwgMTIzLCA0NTYsIDc4OSwg'
        'MjAyNCwgNywgMTMsIDIxLCAzMywgNTUsCiAgICAgICAgIDc3LCA5OSwgMTExLCAyMjIsIDMzMywg'
        'NDQ0LCA1NTUsIDc3NywgODg4LCA5OTldCkZFQVRVUkVTICAgICAgID0gWydTTlInLCAnUlNTSScs'
        'ICdQRFInLCAnU3BlZWQnLCAnUmVsYXRpdmVfU3BlZWQnXQpXSU5ET1dfU0laRVMgICA9IFsxLCA1'
        'LCAxNV0KRVBJU09ERV9MRU5HVEggPSA2MCAgICAgICAgICAjIHNlY29uZHMgcGVyIHBzZXVkby1l'
        'cGlzb2RlCldJTkRPV19TRUMgICAgID0gMTUgICAgICAgICAgIyB3aW5kb3cgdXNlZCBieSBSRi1T'
        'dGF0IC8gWEdCb29zdC1vbmx5IC8gTFNUTQpLX05FSUdIQk9SUyAgICA9IDUKU1BFRURfUkFOR0Ug'
        'ICAgPSB7MTU6ICgxMywgMTcpLCAyNTogKDIzLCAyNyl9ClNDRU5BUklPX1RPX0xBQkVMID0gezE6'
        'IDAsIDI6IDEsIDM6IDJ9ICAgIyBJbnRlcmZlcmVuY2UgLyBSZWFjdGl2ZSAvIENvbnN0YW50Cgoj'
        'IEVESVQgVEhFU0UgdG8gcG9pbnQgYXQgeW91ciBsb2NhbCBjb3BpZXMgb2YgdGhlIGRhdGFzZXQ6'
        'CkRBVEFTRVRfUEFUSFMgPSB7CiAgICAyNTogb3MuZW52aXJvbi5nZXQoJ0RBVEFTRVRfMjUnLCAn'
        'RGF0YXNldF8xLmNzdicpLAogICAgMTU6IG9zLmVudmlyb24uZ2V0KCdEQVRBU0VUXzE1JywgJ0Rh'
        'dGFzZXRfMi5jc3YnKSwKfQoKCiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSACiMgRGF0YXNldCBsb2FkaW5nICAobWlycm9ycyBub3RlYm9vayBDZWxsIDIp'
        'CiMg4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACmRlZiBs'
        'b2FkX2RhdGFzZXQodGFyZ2V0X3NwZWVkLCBwYXRoPU5vbmUpOgogICAgIiIiTG9hZCBDU1YsIG1h'
        'cCBsYWJlbHMsIGJ1aWxkIHBzZXVkb19ydW5faWQsIGZpbHRlciB0byBvbmUgc3BlZWQgYmFuZC4i'
        'IiIKICAgIGlmIHBhdGggaXMgTm9uZToKICAgICAgICBwYXRoID0gREFUQVNFVF9QQVRIU1t0YXJn'
        'ZXRfc3BlZWRdCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMocGF0aCk6CiAgICAgICAgcmFpc2Ug'
        'RmlsZU5vdEZvdW5kRXJyb3IoCiAgICAgICAgICAgIGYiRGF0YXNldCBmb3Ige3RhcmdldF9zcGVl'
        'ZH0gbS9zIG5vdCBmb3VuZCBhdCAne3BhdGh9Jy4gIgogICAgICAgICAgICBmIlNldCBEQVRBU0VU'
        'X1BBVEhTW3t0YXJnZXRfc3BlZWR9XSBpbiBjb21tb24ucHkgb3IgdGhlICIKICAgICAgICAgICAg'
        'ZiJEQVRBU0VUX3t0YXJnZXRfc3BlZWR9IGVudmlyb25tZW50IHZhcmlhYmxlLiIpCiAgICBkZiA9'
        'IHBkLnJlYWRfY3N2KHBhdGgpCiAgICBkZlsnbGFiZWwnXSA9IGRmWydTY2VuYXJpbyddLm1hcChT'
        'Q0VOQVJJT19UT19MQUJFTCkKICAgIGRmID0gZGYuc29ydF92YWx1ZXMoWydTcGVlZCcsICdTY2Vu'
        'YXJpbycsICdUaW1lJ10pLnJlc2V0X2luZGV4KGRyb3A9VHJ1ZSkKICAgIGRmWydwc2V1ZG9fcnVu'
        'X2lkJ10gPSAoCiAgICAgICAgZGZbJ1NwZWVkJ10ucm91bmQoMikuYXN0eXBlKHN0cikgKyAnXycg'
        'KwogICAgICAgIGRmWydTY2VuYXJpbyddLmFzdHlwZShzdHIpICsgJ18nICsKICAgICAgICAoZGZb'
        'J1RpbWUnXSAvLyBFUElTT0RFX0xFTkdUSCkuYXN0eXBlKGludCkuYXN0eXBlKHN0cikKICAgICkK'
        'ICAgIGxvLCBoaSA9IFNQRUVEX1JBTkdFW3RhcmdldF9zcGVlZF0KICAgIGRmX3NwZWVkID0gZGZb'
        'KGRmWydTcGVlZCddID49IGxvKSAmIChkZlsnU3BlZWQnXSA8PSBoaSldLmNvcHkoKQogICAgcmV0'
        'dXJuIGRmX3NwZWVkCgoKZGVmIHNldF9hbGxfc2VlZHMoc2VlZCk6CiAgICBucC5yYW5kb20uc2Vl'
        'ZChzZWVkKQogICAgcmFuZG9tLnNlZWQoc2VlZCkKICAgIG9zLmVudmlyb25bJ1BZVEhPTkhBU0hT'
        'RUVEJ10gPSBzdHIoc2VlZCkKICAgIHRyeToKICAgICAgICBpbXBvcnQgdGVuc29yZmxvdyBhcyB0'
        'ZgogICAgICAgIHRmLnJhbmRvbS5zZXRfc2VlZChzZWVkKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoK'
        'ICAgICAgICBwYXNzCgoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIAKIyBFcGlzb2RlIHNwbGl0ICAoVkVSQkFUSU0gZnJvbSBub3RlYm9vaykKIyDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKZGVmIHNwbGl0X2J5'
        'X2VwaXNvZGUoZGYsIHRyYWluX3JhdGlvPTAuNiwgdmFsX3JhdGlvPTAuMiwgc2VlZD00Mik6CiAg'
        'ICAiIiJTdHJhdGlmaWVkIGVwaXNvZGUtYmFzZWQgc3BsaXQgKHBlci1jbGFzcyBzaHVmZmxlKS4i'
        'IiIKICAgIHJuZyA9IG5wLnJhbmRvbS5SYW5kb21TdGF0ZShzZWVkKQogICAgZXBfdG9fbGFiZWwg'
        'PSAoZGYuZ3JvdXBieSgncHNldWRvX3J1bl9pZCcpWydsYWJlbCddCiAgICAgICAgICAgICAgICAg'
        'ICAgIC5hZ2cobGFtYmRhIHg6IHguaWxvY1swXSkudG9fZGljdCgpKQogICAgZXBzX2J5X2NsYXNz'
        'ID0ge30KICAgIGZvciBlcCwgbGJsIGluIGVwX3RvX2xhYmVsLml0ZW1zKCk6CiAgICAgICAgZXBz'
        'X2J5X2NsYXNzLnNldGRlZmF1bHQobGJsLCBbXSkuYXBwZW5kKGVwKQoKICAgIHRyYWluX2Vwcywg'
        'dmFsX2VwcywgdGVzdF9lcHMgPSBbXSwgW10sIFtdCiAgICBmb3IgbGJsLCBlcHMgaW4gZXBzX2J5'
        'X2NsYXNzLml0ZW1zKCk6CiAgICAgICAgZXBzID0gbnAuYXJyYXkoZXBzKQogICAgICAgIHJuZy5z'
        'aHVmZmxlKGVwcykKICAgICAgICBuID0gbGVuKGVwcykKICAgICAgICBuX3RyYWluID0gaW50KHJv'
        'dW5kKHRyYWluX3JhdGlvICogbikpCiAgICAgICAgbl92YWwgPSBpbnQocm91bmQodmFsX3JhdGlv'
        'ICogbikpCiAgICAgICAgaWYgbiA+PSAzOgogICAgICAgICAgICBuX3RyYWluID0gbWF4KDEsIG1p'
        'bihuIC0gMiwgbl90cmFpbikpCiAgICAgICAgICAgIG5fdmFsID0gbWF4KDEsIG1pbihuIC0gbl90'
        'cmFpbiAtIDEsIG5fdmFsKSkKICAgICAgICB0cmFpbl9lcHMuZXh0ZW5kKGVwc1s6bl90cmFpbl0u'
        'dG9saXN0KCkpCiAgICAgICAgdmFsX2Vwcy5leHRlbmQoZXBzW25fdHJhaW46bl90cmFpbiArIG5f'
        'dmFsXS50b2xpc3QoKSkKICAgICAgICB0ZXN0X2Vwcy5leHRlbmQoZXBzW25fdHJhaW4gKyBuX3Zh'
        'bDpdLnRvbGlzdCgpKQoKICAgIHRyYWluX2RmID0gZGZbZGZbJ3BzZXVkb19ydW5faWQnXS5pc2lu'
        'KHRyYWluX2VwcyldLmNvcHkoKQogICAgdmFsX2RmICAgPSBkZltkZlsncHNldWRvX3J1bl9pZCdd'
        'LmlzaW4odmFsX2VwcyldLmNvcHkoKQogICAgdGVzdF9kZiAgPSBkZltkZlsncHNldWRvX3J1bl9p'
        'ZCddLmlzaW4odGVzdF9lcHMpXS5jb3B5KCkKICAgIGFzc2VydCBub3QgKHNldCh0cmFpbl9lcHMp'
        'ICYgc2V0KHRlc3RfZXBzKSksICdMRUFLQUdFIScKICAgIGFzc2VydCBub3QgKHNldCh0cmFpbl9l'
        'cHMpICYgc2V0KHZhbF9lcHMpKSwgICdMRUFLQUdFIScKICAgIGFzc2VydCBub3QgKHNldCh2YWxf'
        'ZXBzKSAgICYgc2V0KHRlc3RfZXBzKSksICdMRUFLQUdFIScKICAgIHJldHVybiB0cmFpbl9kZiwg'
        'dmFsX2RmLCB0ZXN0X2RmCgoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIAKIyBXaW5kb3dpbmcgIChWRVJCQVRJTSDigJQgbm9uLW92ZXJsYXBwaW5nLCBz'
        'dGVwID0gd2luZG93IGxlbmd0aCkKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIAKZGVmIGNyZWF0ZV93aW5kb3dzKGRmLCB3aW5kb3dfc2VjLCBzdGVwX3Nl'
        'Yz1Ob25lKToKICAgIGlmIHN0ZXBfc2VjIGlzIE5vbmU6CiAgICAgICAgc3RlcF9zZWMgPSAxICAj'
        'IG92ZXJsYXBwaW5nIOKAlCBpZGVudGlrIGRlbmdhbiBEZXRlY3RvciBub3RlYm9vayAocnVuX2xz'
        'dG1fcmVsdV9pb2wxKQogICAgWF9saXN0LCB5X2xpc3QsIGVwX2xpc3QgPSBbXSwgW10sIFtdCiAg'
        'ICBmb3IgcnVuX2lkLCBncm91cCBpbiBkZi5ncm91cGJ5KCdwc2V1ZG9fcnVuX2lkJyk6CiAgICAg'
        'ICAgZ3JvdXAgPSBncm91cC5zb3J0X3ZhbHVlcygnVGltZScpLnJlc2V0X2luZGV4KGRyb3A9VHJ1'
        'ZSkKICAgICAgICB0aW1lcyA9IGdyb3VwWydUaW1lJ10udmFsdWVzCiAgICAgICAgWF9yYXcgPSBn'
        'cm91cFtGRUFUVVJFU10udmFsdWVzCiAgICAgICAgeV9yYXcgPSBncm91cFsnbGFiZWwnXS52YWx1'
        'ZXMKICAgICAgICB0MCwgdF9lbmQgPSB0aW1lc1swXSwgdGltZXNbLTFdCiAgICAgICAgd2hpbGUg'
        'dDAgKyB3aW5kb3dfc2VjIDw9IHRfZW5kOgogICAgICAgICAgICB0MSA9IHQwICsgd2luZG93X3Nl'
        'YwogICAgICAgICAgICBtYXNrID0gKHRpbWVzID49IHQwKSAmICh0aW1lcyA8IHQxKQogICAgICAg'
        'ICAgICBpZiBtYXNrLnN1bSgpID4gMDoKICAgICAgICAgICAgICAgIFhfbGlzdC5hcHBlbmQoWF9y'
        'YXdbbWFza10pCiAgICAgICAgICAgICAgICB5X2xpc3QuYXBwZW5kKG5wLmJpbmNvdW50KHlfcmF3'
        'W21hc2tdKS5hcmdtYXgoKSkKICAgICAgICAgICAgICAgIGVwX2xpc3QuYXBwZW5kKHJ1bl9pZCkK'
        'ICAgICAgICAgICAgdDAgKz0gc3RlcF9zZWMKICAgIHJldHVybiBYX2xpc3QsIG5wLmFycmF5KHlf'
        'bGlzdCksIGVwX2xpc3QKCgpkZWYgcGFkX3NlcXVlbmNlcyhYX2xpc3QsIG1heF9sZW49Tm9uZSk6'
        'CiAgICBpZiBtYXhfbGVuIGlzIE5vbmU6CiAgICAgICAgbWF4X2xlbiA9IG1heChsZW4oeCkgZm9y'
        'IHggaW4gWF9saXN0KQogICAgbl9mZWF0ID0gWF9saXN0WzBdLnNoYXBlWzFdCiAgICBvdXQgPSBu'
        'cC56ZXJvcygobGVuKFhfbGlzdCksIG1heF9sZW4sIG5fZmVhdCksIGR0eXBlPW5wLmZsb2F0MzIp'
        'CiAgICBmb3IgaSwgeCBpbiBlbnVtZXJhdGUoWF9saXN0KToKICAgICAgICBzbCA9IG1pbihsZW4o'
        'eCksIG1heF9sZW4pCiAgICAgICAgb3V0W2ksIDpzbCwgOl0gPSB4WzpzbF0KICAgIHJldHVybiBv'
        'dXQsIG1heF9sZW4KCgpkZWYgcGFkX2FuZF9ub3JtYWxpemUoWF90cmFpbl9saXN0LCBYX3ZhbF9s'
        'aXN0LCBYX3Rlc3RfbGlzdCk6CiAgICBtYXhfbGVuID0gbWF4KGxlbih4KSBmb3IgeCBpbiBYX3Ry'
        'YWluX2xpc3QpCiAgICBuX2ZlYXR1cmVzID0gWF90cmFpbl9saXN0WzBdLnNoYXBlWzFdCiAgICBk'
        'ZWYgcGFkKFhfbGlzdCk6CiAgICAgICAgb3V0ID0gbnAuemVyb3MoKGxlbihYX2xpc3QpLCBtYXhf'
        'bGVuLCBuX2ZlYXR1cmVzKSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICBmb3IgaSwgeCBpbiBl'
        'bnVtZXJhdGUoWF9saXN0KToKICAgICAgICAgICAgc2wgPSBtaW4obGVuKHgpLCBtYXhfbGVuKQog'
        'ICAgICAgICAgICBvdXRbaSwgOnNsLCA6XSA9IHhbOnNsXQogICAgICAgIHJldHVybiBvdXQKICAg'
        'IFh0ciwgWHYsIFh0ZSA9IHBhZChYX3RyYWluX2xpc3QpLCBwYWQoWF92YWxfbGlzdCksIHBhZChY'
        'X3Rlc3RfbGlzdCkKICAgIHNjID0gU3RhbmRhcmRTY2FsZXIoKQogICAgWHRyID0gc2MuZml0X3Ry'
        'YW5zZm9ybShYdHIucmVzaGFwZSgtMSwgbl9mZWF0dXJlcykpLnJlc2hhcGUoWHRyLnNoYXBlKQog'
        'ICAgWHYgID0gc2MudHJhbnNmb3JtKFh2LnJlc2hhcGUoLTEsIG5fZmVhdHVyZXMpKS5yZXNoYXBl'
        'KFh2LnNoYXBlKQogICAgWHRlID0gc2MudHJhbnNmb3JtKFh0ZS5yZXNoYXBlKC0xLCBuX2ZlYXR1'
        'cmVzKSkucmVzaGFwZShYdGUuc2hhcGUpCiAgICByZXR1cm4gWHRyLCBYdiwgWHRlLCBtYXhfbGVu'
        'CgoKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAKIyBU'
        'YWJ1bGFyIGZlYXR1cmVzICAoVkVSQkFUSU0g4oCUIDcgc3RhdHMgeCA1IGZlYXR1cmVzID0gMzUg'
        'ZGltcykKIyDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDi'
        'lIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIDilIAK'
        'ZGVmIGV4dHJhY3RfdGFidWxhcl9mZWF0dXJlcyhYX3BhZGRlZCk6CiAgICBvdXQgPSBbXQogICAg'
        'Zm9yIHggaW4gWF9wYWRkZWQ6CiAgICAgICAgbSA9IH4oeCA9PSAwKS5hbGwoYXhpcz0xKQogICAg'
        'ICAgIHh2ID0geFttXSBpZiBtLnN1bSgpID4gMCBlbHNlIHgKICAgICAgICBmID0gW10KICAgICAg'
        'ICBmb3IgYyBpbiByYW5nZSh4di5zaGFwZVsxXSk6CiAgICAgICAgICAgIHYgPSB4dls6LCBjXQog'
        'ICAgICAgICAgICBmLmV4dGVuZChbbnAubWVhbih2KSwgbnAuc3RkKHYpLCBucC5taW4odiksIG5w'
        'Lm1heCh2KSwKICAgICAgICAgICAgICAgICAgICAgIG5wLm1lZGlhbih2KSwgbnAucGVyY2VudGls'
        'ZSh2LCAyNSksIG5wLnBlcmNlbnRpbGUodiwgNzUpXSkKICAgICAgICBvdXQuYXBwZW5kKGYpCiAg'
        'ICByZXR1cm4gbnAuYXJyYXkob3V0KQoKCiMgNyBzdW1tYXJ5IHN0YXRpc3RpY3MgcGVyIGZlYXR1'
        'cmUsIGluIHRoZSBvcmRlciBwcm9kdWNlZCBhYm92ZToKU1RBVFNfUEVSX0ZFQVRVUkUgPSA3CmRl'
        'ZiBmZWF0dXJlX2Jsb2NrX2luZGljZXMoZmVhdHVyZV9uYW1lKToKICAgICIiIlJldHVybiB0aGUg'
        'NyBjb2x1bW4gaW5kaWNlcyBpbiB0aGUgMzUtZGltIHZlY3RvciBmb3Igb25lIHJhdyBmZWF0dXJl'
        'LiIiIgogICAgYyA9IEZFQVRVUkVTLmluZGV4KGZlYXR1cmVfbmFtZSkKICAgIHJldHVybiBsaXN0'
        'KHJhbmdlKGMgKiBTVEFUU19QRVJfRkVBVFVSRSwgKGMgKyAxKSAqIFNUQVRTX1BFUl9GRUFUVVJF'
        'KSkKCgpkZWYgc2VsZWN0X3RhYnVsYXJfY29sdW1ucyhYMzUsIGRyb3BfZmVhdHVyZXM9KCkpOgog'
        'ICAgIiIiUmV0dXJuIGEgY29weSBvZiB0aGUgMzUtZGltIHRhYnVsYXIgbWF0cml4IHdpdGggdGhl'
        'IGNvbHVtbnMgYmVsb25naW5nCiAgICB0byBgZHJvcF9mZWF0dXJlc2AgcmVtb3ZlZCAodXNlZCBm'
        'b3IgdGhlIG5vLXJlbGF0aXZlLXNwZWVkIGFibGF0aW9ucykuIiIiCiAgICBkcm9wID0gc2V0KCkK'
        'ICAgIGZvciBmbiBpbiBkcm9wX2ZlYXR1cmVzOgogICAgICAgIGRyb3AudXBkYXRlKGZlYXR1cmVf'
        'YmxvY2tfaW5kaWNlcyhmbikpCiAgICBrZWVwID0gW2kgZm9yIGkgaW4gcmFuZ2UoWDM1LnNoYXBl'
        'WzFdKSBpZiBpIG5vdCBpbiBkcm9wXQogICAgcmV0dXJuIFgzNVs6LCBrZWVwXSwga2VlcAoKCiMg'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA'
        '4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSA4pSACiMgTWV0cmlj'
        'cyAgKFZFUkJBVElNKQojIOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKUgOKU'
        'gOKUgOKUgApkZWYgY29tcHV0ZV9tZXRyaWNzKHlfdHJ1ZSwgeV9wcmVkLCBudW1fY2xhc3Nlcz0z'
        'KToKICAgIGFjYyA9IGZsb2F0KGFjY3VyYWN5X3Njb3JlKHlfdHJ1ZSwgeV9wcmVkKSkKICAgIHAs'
        'IHIsIGYxLCBfID0gcHJlY2lzaW9uX3JlY2FsbF9mc2NvcmVfc3VwcG9ydCgKICAgICAgICB5X3Ry'
        'dWUsIHlfcHJlZCwgYXZlcmFnZT0nbWFjcm8nLCB6ZXJvX2RpdmlzaW9uPTApCiAgICBsYWJlbHMg'
        'PSBsaXN0KHJhbmdlKG51bV9jbGFzc2VzKSkKICAgIHBfcGMsIHJfcGMsIGYxX3BjLCBfID0gcHJl'
        'Y2lzaW9uX3JlY2FsbF9mc2NvcmVfc3VwcG9ydCgKICAgICAgICB5X3RydWUsIHlfcHJlZCwgbGFi'
        'ZWxzPWxhYmVscywgYXZlcmFnZT1Ob25lLCB6ZXJvX2RpdmlzaW9uPTApCiAgICBjbSA9IGNvbmZ1'
        'c2lvbl9tYXRyaXgoeV90cnVlLCB5X3ByZWQsIGxhYmVscz1sYWJlbHMpLnRvbGlzdCgpCiAgICBy'
        'ZXR1cm4gZGljdChhY2N1cmFjeT1hY2MsIHByZWNpc2lvbl9tYWNybz1mbG9hdChwKSwKICAgICAg'
        'ICAgICAgICAgIHJlY2FsbF9tYWNybz1mbG9hdChyKSwgZjFfbWFjcm89ZmxvYXQoZjEpLAogICAg'
        'ICAgICAgICAgICAgcHJlY2lzaW9uX3Blcl9jbGFzcz1wX3BjLnRvbGlzdCgpLAogICAgICAgICAg'
        'ICAgICAgcmVjYWxsX3Blcl9jbGFzcz1yX3BjLnRvbGlzdCgpLAogICAgICAgICAgICAgICAgZjFf'
        'cGVyX2NsYXNzPWYxX3BjLnRvbGlzdCgpLAogICAgICAgICAgICAgICAgY29uZnVzaW9uX21hdHJp'
        'eD1jbSkKCgpkZWYgYWdnKHZhbHVlcywgcGN0PVRydWUpOgogICAgIiIibWVhbiDCsSBzYW1wbGUt'
        'c3RkIChkZG9mPTEpIGhlbHBlciwgbWF0Y2hpbmcgdGhlIG5vdGVib29rcy4iIiIKICAgIGEgPSBu'
        'cC5hc2FycmF5KHZhbHVlcywgZHR5cGU9ZmxvYXQpCiAgICBtLCBzID0gYS5tZWFuKCksIChhLnN0'
        'ZChkZG9mPTEpIGlmIGxlbihhKSA+IDEgZWxzZSAwLjApCiAgICBpZiBwY3Q6CiAgICAgICAgbSwg'
        'cyA9IG0gKiAxMDAsIHMgKiAxMDAKICAgIHJldHVybiBtLCBzCgoKQ0xBU1NfTkFNRVMgPSBbJ0lu'
        'dGVyZmVyZW5jZScsICdSZWFjdGl2ZScsICdDb25zdGFudCddCg=='
)
_src = base64.b64decode(_b64.encode('ascii')).decode('utf-8')
assert 'step_sec = 1' in _src, 'step_sec BUKAN 1!'
with open('/content/common.py', 'w') as _f:
    _f.write(_src)
print('✅ common.py ditulis — step_sec=1 (pipeline Detector)')

if '/content' not in sys.path:
    sys.path.insert(0, '/content')
print('✅ Selesai — lanjut ke cell berikutnya')


ℹ️  Belum ada cache
✅ common.py ditulis — step_sec=1 (pipeline Detector)
✅ Selesai — lanjut ke cell berikutnya


In [5]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 3 — (DIUBAH untuk 20 seed)
# Injeksi hasil lama (67.78 / 51.12) DIHAPUS karena itu hasil 5-seed asli
# dan TIDAK bisa dipakai untuk 20 seed (jumlah nilai tidak cocok, dan
# 15 seed tambahan tidak pernah dijalankan untuk baseline ini).
#
# Solusi: variant 'baseline' sekarang dijalankan lewat run_variant() yang
# sama seperti variant lain (konfigurasinya memang identik: activation=relu,
# l1_lambda=1e-4, patience=15, clipnorm=None -- lihat VARIANTS di bawah),
# jadi otomatis konsisten untuk berapa pun jumlah SEEDS.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('ℹ️  Baseline LSTM-RELU-IO-L1 akan dihitung ulang penuh untuk 20 seed '
      '(bukan pakai nilai lama 5-seed).')


  25m/s: 67.78 ± 27.01%
  15m/s: 51.12 ± 28.03%
✅ Baseline Detector siap


In [6]:
"""
t3_lstm_sensitivity.py  —  T3 (review item C3)

The released code reports LSTM-RELU-IO-L1 as severely unstable (per-seed std
~27-29 pp) and defers it to Appendix A. A reviewer will ask whether the
instability is intrinsic or an artefact of un-tuned settings. This script runs
the baseline plus four stabilisation variants across all 20 seeds at both
regimes, so Appendix A can report "we attempted stabilisation; here is what
happened" instead of only documenting the failure.

Variants:
    baseline      : as released (cell activation ReLU, L1=1e-4, patience 15)
    tanh          : cell activation tanh (restores bounded recurrent output)
    clipnorm      : ReLU + gradient clipping (clipnorm=1.0)
    low_l1        : ReLU + reduced L1 (1e-5)
    long_patience : ReLU + early-stopping patience 30

For each variant we report mean ± std accuracy and, crucially, the per-seed
spread (the whole point is whether the variance shrinks).

NEEDS THE DATASET and TensorFlow.
"""
import os, json
import numpy as np
from sklearn.preprocessing import StandardScaler
from common import (load_dataset, split_by_episode, create_windows,
                    pad_sequences, compute_metrics, agg, set_all_seeds,
                    SEEDS, WINDOW_SEC, FEATURES)

In [7]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l1
from tensorflow.keras.initializers import GlorotUniform

In [8]:
VARIANTS = {
    'baseline':      dict(activation='relu', l1_lambda=1e-4, patience=15, clipnorm=None),
    'tanh':          dict(activation='tanh', l1_lambda=1e-4, patience=15, clipnorm=None),
    'clipnorm':      dict(activation='relu', l1_lambda=1e-4, patience=15, clipnorm=1.0),
    'low_l1':        dict(activation='relu', l1_lambda=1e-5, patience=15, clipnorm=None),
    'long_patience': dict(activation='relu', l1_lambda=1e-4, patience=30, clipnorm=None),
}
LSTM_EPOCHS = 100
BATCH_SIZE  = 32

In [9]:
def build_model(input_shape, activation, l1_lambda, seed):
    tf.keras.utils.set_random_seed(seed)
    init = GlorotUniform(seed=seed)
    inputs = Input(shape=input_shape)
    x = Dense(input_shape[-1], activation='relu',
              kernel_regularizer=l1(l1_lambda), kernel_initializer=init)(inputs)
    x = LSTM(128, return_sequences=True, activation=activation,
             kernel_initializer=init)(x)
    x = Dropout(0.3)(x)
    x = LSTM(64, return_sequences=False, activation=activation,
             kernel_initializer=init)(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu', kernel_initializer=init)(x)
    outputs = Dense(3, activation='softmax',
                    kernel_regularizer=l1(l1_lambda), kernel_initializer=init)(x)
    return Model(inputs, outputs)

In [10]:
def run_variant(df_speed, seed, cfg):
    set_all_seeds(seed)
    tr, va, te = split_by_episode(df_speed, seed=seed)
    Xtr_l, ytr, _ = create_windows(tr, WINDOW_SEC)
    Xv_l,  yv,  _ = create_windows(va, WINDOW_SEC)
    Xte_l, yte, _ = create_windows(te, WINDOW_SEC)
    if min(len(Xtr_l), len(Xv_l), len(Xte_l)) == 0:
        return None
    max_len = max(max(len(x) for x in Xtr_l),
                  max(len(x) for x in Xv_l),
                  max(len(x) for x in Xte_l))
    Xtr, _ = pad_sequences(Xtr_l, max_len)
    Xv,  _ = pad_sequences(Xv_l,  max_len)
    Xte, _ = pad_sequences(Xte_l, max_len)
    n_feat = len(FEATURES)
    sc = StandardScaler()
    Xtr = sc.fit_transform(Xtr.reshape(-1, n_feat)).reshape(Xtr.shape)
    Xv  = sc.transform(Xv.reshape(-1, n_feat)).reshape(Xv.shape)
    Xte = sc.transform(Xte.reshape(-1, n_feat)).reshape(Xte.shape)

    model = build_model((max_len, n_feat), cfg['activation'], cfg['l1_lambda'], seed)
    opt = Adam(1e-3, clipnorm=cfg['clipnorm']) if cfg['clipnorm'] else Adam(1e-3)
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    cb = [EarlyStopping(monitor='val_loss', patience=cfg['patience'],
                        restore_best_weights=True),
          ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)]
    model.fit(Xtr, ytr, validation_data=(Xv, yv), epochs=LSTM_EPOCHS,
              batch_size=BATCH_SIZE, callbacks=cb, verbose=0)
    y_pred = np.argmax(model.predict(Xte, verbose=0), axis=1)
    acc = compute_metrics(yte, y_pred)['accuracy']
    tf.keras.backend.clear_session()
    return acc

In [11]:
def main(outdir='outputs_lstm_sensitivity'):
    os.makedirs(outdir, exist_ok=True)
    results = {}
    for spd in (25, 15):
        df = load_dataset(spd)
        for vname, cfg in VARIANTS.items():
            accs = []
            for seed in SEEDS:
                a = run_variant(df, seed, cfg)
                if a is not None:
                    accs.append(a)
                    print(f'  [{spd}m {vname:13s} seed {seed}] acc={a:.4f}')
            m, s = agg(accs)
            results[f'{spd}ms_{vname}'] = {
                'speed': spd, 'variant': vname, 'per_seed': accs,
                'mean': round(m, 2), 'std_pp': round(s, 2)}

    print('\n' + '=' * 70)
    print('  T3 — LSTM-RELU-IO-L1 stabilisation sensitivity (acc %, 20 seeds)')
    print('=' * 70)
    print(f'{"Regime":>8s}{"Variant":>16s}{"Mean":>10s}{"Std (pp)":>12s}')
    print('-' * 70)
    for k, v in results.items():
        print(f'{v["speed"]:>6d}m{v["variant"]:>16s}{v["mean"]:>10.2f}{v["std_pp"]:>12.2f}')

    with open(os.path.join(outdir, 't3_lstm_sensitivity.json'), 'w') as f:
        json.dump(results, f, indent=2)
    print(f'\nSaved to {outdir}/   (lower Std => stabilisation worked)')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 4 — Guard: pastikan dataset path benar SEBELUM main()
#   (mencegah FileNotFoundError yang membingungkan kalau lupa
#    re-run cell mount-drive setelah kernel restart di Step 1)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os as _os

_p25 = _os.environ.get('DATASET_25')
_p15 = _os.environ.get('DATASET_15')

assert _p25 and _os.path.exists(_p25), (
    f"DATASET_25 belum ter-set dengan benar (nilai saat ini: {_p25!r}).\n"
    "Kemungkinan besar kamu lupa RE-RUN cell mount-drive (yang isinya "
    "os.environ['DATASET_25'] = ...) setelah kernel restart di Step 1.\n"
    "Scroll ke atas, jalankan ulang cell mount-drive itu, baru lanjut lagi ke sini."
)
assert _p15 and _os.path.exists(_p15), (
    f"DATASET_15 belum ter-set dengan benar (nilai saat ini: {_p15!r}).\n"
    "Kemungkinan besar kamu lupa RE-RUN cell mount-drive setelah kernel restart di Step 1.\n"
    "Scroll ke atas, jalankan ulang cell mount-drive itu, baru lanjut lagi ke sini."
)
print(f'✅ DATASET_25 OK: {_p25}')
print(f'✅ DATASET_15 OK: {_p15}')


In [12]:
main()


   baseline      seed 42] acc=0.7059  [Detector]
   baseline      seed 123] acc=0.6923  [Detector]
   baseline      seed 456] acc=0.7407  [Detector]
   baseline      seed 789] acc=0.2500  [Detector]
   baseline      seed 2024] acc=1.0000  [Detector]
   tanh          seed 42] acc=0.9368
   tanh          seed 123] acc=0.9925


   tanh          seed 456] acc=1.0000
   tanh          seed 789] acc=0.9775


   tanh          seed 2024] acc=0.9833
   clipnorm      seed 42] acc=0.9655
   clipnorm      seed 123] acc=0.9925
   clipnorm      seed 456] acc=1.0000
   clipnorm      seed 789] acc=0.9775
   clipnorm      seed 2024] acc=1.0000
   low_l1        seed 42] acc=0.9598
   low_l1        seed 123] acc=0.9925
   low_l1        seed 456] acc=1.0000
   low_l1        seed 789] acc=0.9775
   low_l1        seed 2024] acc=0.9917
   long_patience seed 42] acc=0.9483
   long_patience seed 123] acc=0.9925
   long_patience seed 456] acc=0.9931
   long_patience seed 789] acc=1.0000
   long_patience seed 2024] acc=1.0000
   baseline      seed 42] acc=0.7500  [Detector]
   baseline      seed 123] acc=0.6364  [Detector]
   baseline      seed 456] acc=0.7368  [Detector]
   baseline      seed 789] acc=0.1250  [Detector]
   baseline      seed 2024] acc=0.3077  [Detector]
   tanh          seed 42] acc=0.9762
   tanh          seed 123] acc=0.8960
   tanh          seed 456] acc=0.9109
   tanh          seed 789] a